# 16 — Prefect: Orkestrering av AI-pipelines

**Fase:** 3 — Dataengineering | **Tid:** 2 timer | **Krav:** Notatbok 14

**Hva du bygger:** En automatisert AI-datapipeline med Prefect — med retry-logikk, logging, scheduled kjøring og betinget LLM-berikelse.

---

## Hva er orkestrering?

```
Uten orkestrering:  python pipeline.py   (kjør manuelt, håp at det ikke feiler)
Med Prefect:        schedule + retry + logging + alerts + UI
```

**Prefect** er et orkestreringverktøy for Python-pipelines. Det er gratis å kjøre lokalt.

Nøkkelbegreper:
- **Flow** — Hele pipelinen (en funksjon med `@flow`)
- **Task** — Et steg i pipelinen (en funksjon med `@task`)
- **Deployment** — Scheduled kjøring av en flow
- **Run** — Én kjøring av en flow

In [ ]:
%pip install -q prefect duckdb pydantic requests

---

## Del 1: Enkelt flow med tasks

In [ ]:
from prefect import flow, task, get_run_logger
from prefect.tasks import task_input_hash
from datetime import timedelta
import requests
import duckdb

@task(
    retries=3,                                  # Prøv 3 ganger ved feil
    retry_delay_seconds=5,                      # Vent 5 sek mellom forsøk
    cache_key_fn=task_input_hash,               # Cache resultatet
    cache_expiration=timedelta(hours=1),
)
def hent_dokumenter(url: str, maks: int = 5) -> list[dict]:
    """Extract: Hent dokumenter fra API."""
    logger = get_run_logger()
    logger.info(f"Henter {maks} dokumenter fra {url}")
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()[:maks]
    logger.info(f"Hentet {len(data)} dokumenter")
    return data

@task
def rens_dokumenter(rådata: list[dict]) -> list[dict]:
    """Transform: Rens og valider."""
    logger = get_run_logger()
    rensede = []
    for dok in rådata:
        body = dok.get("body", "").strip()
        if len(body) >= 20:
            rensede.append({
                "id":      dok["id"],
                "tittel":  dok.get("title", "").strip(),
                "innhold": body,
                "tegn":    len(body),
            })
    logger.info(f"{len(rådata)} inn → {len(rensede)} gyldige")
    return rensede

@task
def lagre_til_database(dokumenter: list[dict], db_sti: str = "prefect_demo.duckdb") -> int:
    """Load: Lagre til DuckDB."""
    logger = get_run_logger()
    con = duckdb.connect(db_sti)
    con.execute("""
        CREATE TABLE IF NOT EXISTS dokumenter (
            id INTEGER, tittel VARCHAR, innhold VARCHAR, tegn INTEGER
        )
    """)
    con.execute("DELETE FROM dokumenter")
    for dok in dokumenter:
        con.execute("INSERT INTO dokumenter VALUES (?, ?, ?, ?)",
                    [dok["id"], dok["tittel"], dok["innhold"], dok["tegn"]])
    antall = con.execute("SELECT COUNT(*) FROM dokumenter").fetchone()[0]
    con.close()
    logger.info(f"Lastet {antall} dokumenter til {db_sti}")
    return antall

@flow(name="ETL Pipeline")
def etl_pipeline(url: str = "https://jsonplaceholder.typicode.com/posts") -> dict:
    """Hovedflow: Koordinerer alle tasks."""
    rådata    = hent_dokumenter(url, maks=8)
    rensede   = rens_dokumenter(rådata)
    antall    = lagre_til_database(rensede)
    return {"hentet": len(rådata), "lastet": antall}

print("Flow definert.")

In [ ]:
# Kjør flowen
resultat = etl_pipeline()
print(f"\nResultat: {resultat}")

---

## Del 2: Betinget LLM-berikelse

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

@task(retries=2, retry_delay_seconds=3)
def klassifiser_med_llm(dokumenter: list[dict]) -> list[dict]:
    """
    Berik dokumenter med LLM-generert kategori.
    Kjøres bare på dokumenter uten eksisterende kategori.
    """
    logger = get_run_logger()
    beriket = []
    for dok in dokumenter:
        svar = llm.chat.completions.create(
            model="llama3.2",
            messages=[{
                "role": "user",
                "content": f"Klassifiser dette i én av: [teknisk, juridisk, finansiell, annet]. Kun kategorinavn.\n\n{dok['tittel']}"
            }],
            temperature=0,
            max_tokens=10,
        )
        kategori = svar.choices[0].message.content.strip().lower()
        beriket.append({**dok, "kategori": kategori})

    logger.info(f"Beriket {len(beriket)} dokumenter")
    return beriket

@flow(name="AI-beriket Pipeline")
def ai_pipeline(
    url: str = "https://jsonplaceholder.typicode.com/posts",
    berik_med_ai: bool = True,
) -> dict:
    rådata  = hent_dokumenter(url, maks=3)
    rensede = rens_dokumenter(rådata)

    if berik_med_ai and rensede:
        rensede = klassifiser_med_llm(rensede)

    antall = lagre_til_database(rensede)
    return {"lastet": antall, "ai_beriket": berik_med_ai}

resultat = ai_pipeline(berik_med_ai=True)
print(f"Resultat: {resultat}")

---

## Del 3: Schedule og Deployment

```python
# Kjør dette i terminal for å schedulere pipelinen:

from prefect.deployments import Deployment
from prefect.server.schemas.schedules import CronSchedule

deployment = Deployment.build_from_flow(
    flow=etl_pipeline,
    name="daglig-etl",
    schedule=CronSchedule(cron="0 6 * * *"),  # Kjør kl. 06:00 hver dag
)
deployment.apply()

# Start Prefect-agent: prefect agent start -q default
# Åpne dashboard: prefect server start
```

---

## Prefect vs. Alternativene

| Verktøy | Gratis? | Styrke |
|---------|---------|-------|
| **Prefect** | ✅ Lokalt | Python-native, enkel oppsett |
| Apache Airflow | ✅ Open source | Industristandard, mer kompleks |
| Dagster | ✅ Community | Datakvalitetsfokus |
| GitHub Actions | ✅ Gratis | CI/CD + enkle schedules |

---

## Oppsummering

| Prefect-konsept | Dekorator | Hva det gjør |
|----------------|-----------|-------------|
| Flow | `@flow` | Koordinerer tasks, gir oversikt |
| Task | `@task` | Enkelt steg, støtter retry/cache |
| Logger | `get_run_logger()` | Strukturert logging |
| Deployment | `Deployment` | Scheduled, reproducerbar kjøring |

---

## Hva er neste steg?

**Fase 4 starter! Neste: `17_docker_for_ai.ipynb`** — Pakk AI-appen din i en container slik at den kjører likt overalt — lokalt, i sky, i CI/CD.